# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tracy030115/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row of the dataset represents one client on one article on the recorded date.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import notebook_login
import pandas as pd
notebook_login()

from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

row = next(iter(ds))
pd.DataFrame([row])


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, scroll_events, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other
Label: None, because it is unsupervised training
Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
Excluded: gsc_sum_position should be excluded since the sum of positions in total is not meaningful in finding performance archetypes

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(ds.features)

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [4]:
# 1. GRAIN
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || report_date) AS distinct_triples
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_triples
0,78835655,78829265


In [5]:
# 2. COUNTS
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_pages,distinct_clients
0,78835655,427292,70


In [9]:
# 3. MISSING VALUES
con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_gsc_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_gsc_clicks,
    FROM (
        SELECT gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,null_gsc_impressions,null_gsc_clicks
0,98006.0,98006.0


In [7]:
# 4. WINDOWS
con.sql(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM (
        SELECT report_date
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date
0,2025-01-27,2026-06-30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell the difference between a page that's actually underperforming and a page that is too new to gain enough impressions. Older pages have had more time to build up impressions and clicks, so they naturally look more popular than newer ones even if the newer content is just as good.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.